# Deepfake Detection — Exploratory Data Analysis

This notebook explores the dataset before training:
- Class distribution
- Sample images from each class
- Image statistics (brightness, contrast, resolution)
- Augmentation preview
- Frequency domain analysis (FFT artifacts)

In [ ]:
import sys
sys.path.append('..')

import os
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Load config
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded ✓')

## 1. Dataset Overview

In [ ]:
from src.dataset import collect_image_paths

real_paths = collect_image_paths('../data/real')
fake_paths = collect_image_paths('../data/fake')

print(f'Real images: {len(real_paths):,}')
print(f'Fake images: {len(fake_paths):,}')
print(f'Total:       {len(real_paths)+len(fake_paths):,}')
print(f'Ratio:       {len(fake_paths)/max(len(real_paths),1):.2f}x fake/real')

In [ ]:
# Class distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = [len(real_paths), len(fake_paths)]
labels = ['Real', 'Fake']
colors = ['#2E86AB', '#E84855']

# Pie chart
axes[0].pie(counts, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 13})
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')

# Bar chart
bars = axes[1].bar(labels, counts, color=colors, edgecolor='white', linewidth=0.5)
for bar, count in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylabel('Number of Images', fontsize=12)
axes[1].set_title('Image Count per Class', fontsize=14, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../results/plots/class_distribution.png', bbox_inches='tight')
plt.show()

## 2. Sample Images

In [ ]:
import random

def show_samples(real_paths, fake_paths, n=8):
    real_sample = random.sample(real_paths, min(n, len(real_paths)))
    fake_sample = random.sample(fake_paths, min(n, len(fake_paths)))
    
    fig, axes = plt.subplots(2, n, figsize=(n*2.5, 6))
    fig.suptitle('Dataset Samples', fontsize=14, fontweight='bold')
    
    for i, (r_path, f_path) in enumerate(zip(real_sample, fake_sample)):
        for ax, path, label, color in [
            (axes[0, i], r_path, 'REAL', '#2E86AB'),
            (axes[1, i], f_path, 'FAKE', '#E84855')
        ]:
            img = cv2.imread(str(path))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img)
            ax.axis('off')
            if i == 0:
                ax.set_title(label, fontsize=12, fontweight='bold', color=color)
    
    plt.tight_layout()
    plt.savefig('../results/plots/sample_images.png', bbox_inches='tight')
    plt.show()

show_samples(real_paths, fake_paths)

## 3. Image Statistics

In [ ]:
def compute_stats(paths, n_sample=200, label=''):
    sample = random.sample(paths, min(n_sample, len(paths)))
    brightness, contrast, heights, widths = [], [], [], []
    
    for p in tqdm(sample, desc=f'  {label}'):
        img = cv2.imread(str(p))
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness.append(gray.mean())
        contrast.append(gray.std())
        heights.append(img.shape[0])
        widths.append(img.shape[1])
    
    return {
        'brightness': brightness, 'contrast': contrast,
        'height': heights, 'width': widths
    }

print('Computing stats...')
real_stats = compute_stats(real_paths, label='Real')
fake_stats = compute_stats(fake_paths, label='Fake')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Image Statistics — Real vs Fake', fontsize=15, fontweight='bold')

stat_pairs = [
    ('brightness', 'Brightness (mean pixel value)'),
    ('contrast',   'Contrast (pixel std dev)'),
    ('height',     'Image Height (px)'),
    ('width',      'Image Width (px)'),
]

for ax, (key, label) in zip(axes.flat, stat_pairs):
    ax.hist(real_stats[key], bins=30, alpha=0.7, color='#2E86AB', label='Real', density=True)
    ax.hist(fake_stats[key], bins=30, alpha=0.7, color='#E84855', label='Fake', density=True)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.legend()
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../results/plots/image_statistics.png', bbox_inches='tight')
plt.show()

## 4. Frequency Domain Analysis (FFT)

GAN-generated deepfakes often leave artifacts in the frequency domain that are invisible to the human eye.

In [ ]:
def compute_avg_fft(paths, n_sample=100):
    sample = random.sample(paths, min(n_sample, len(paths)))
    fft_sum = None
    count = 0
    for p in sample:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (256, 256))
        f = np.fft.fft2(img.astype(np.float32))
        fshift = np.fft.fftshift(f)
        magnitude = np.log(np.abs(fshift) + 1)
        if fft_sum is None:
            fft_sum = magnitude
        else:
            fft_sum += magnitude
        count += 1
    return fft_sum / count if count > 0 else np.zeros((256, 256))

print('Computing FFT averages...')
real_fft = compute_avg_fft(real_paths)
fake_fft = compute_avg_fft(fake_paths)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Average Frequency Spectrum (FFT)', fontsize=14, fontweight='bold')

axes[0].imshow(real_fft, cmap='hot'); axes[0].set_title('Real', fontsize=13); axes[0].axis('off')
axes[1].imshow(fake_fft, cmap='hot'); axes[1].set_title('Fake', fontsize=13); axes[1].axis('off')
diff = np.abs(fake_fft - real_fft)
im = axes[2].imshow(diff, cmap='RdYlGn_r'); axes[2].set_title('Difference (Fake - Real)', fontsize=13); axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig('../results/plots/fft_analysis.png', bbox_inches='tight')
plt.show()
print('Bright spots in the difference image indicate frequency artifacts specific to deepfakes.')

## 5. Augmentation Preview

In [ ]:
from src.dataset import get_train_transforms

transform = get_train_transforms(cfg)

# Pick a random real image
img_path = random.choice(real_paths)
img = cv2.imread(str(img_path))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Show 8 augmentation variants
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Training Augmentation Variants (same image)', fontsize=14, fontweight='bold')

MEAN = np.array(cfg['augmentation']['normalize_mean'])
STD  = np.array(cfg['augmentation']['normalize_std'])

for i, ax in enumerate(axes.flat):
    aug = transform(image=img)
    # Denormalize for display
    t = aug['image'].numpy().transpose(1, 2, 0)
    t = np.clip(t * STD + MEAN, 0, 1)
    ax.imshow(t)
    ax.set_title(f'Variant {i+1}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../results/plots/augmentation_preview.png', bbox_inches='tight')
plt.show()

## 6. Summary

In [ ]:
print('=' * 50)
print('  EDA COMPLETE')
print('=' * 50)
print(f'  Real images  : {len(real_paths):,}')
print(f'  Fake images  : {len(fake_paths):,}')
print(f'  Total        : {len(real_paths)+len(fake_paths):,}')
print(f'  Avg real brightness : {np.mean(real_stats["brightness"]):.1f}')
print(f'  Avg fake brightness : {np.mean(fake_stats["brightness"]):.1f}')
print()
print('  Plots saved to results/plots/')
print('  Ready to train! Run: python src/train.py')